## Stock Signal Analysis — live rules C6-U96-T20-MW30-E5 (point-in-time)
#### Run sentiment_analysis and earnings_date first (run_all.py does this); update the balance sheet monthly

**Scores** (all point-in-time, no look-ahead):
- `Technical_Score`: MA / RSI / MACD / Force Index / OBV / Bollinger / Fibonacci rules.
- `RS_Score`: relative strength, stock vs its sector ETF (60%) and sector ETF vs SPY (40%) over 21/63/126 days, −100..100.
- `Strategy_Score` = 0.5 × Technical + 0.5 × RS (written to `combined_signal`; the app ranks and charts it).
- Fundamentals and sentiment are display only (saved daily to `Reports/factor_history.csv`).

**Live rules = `backtest_engine.WINNER`** (96 stocks, `sector_mapping.EXPANDED_UNIVERSE = "u96"`):
1. **Friday** (week's last session) close: hold the top 10 by `Strategy_Score` (> 0). Walk ranks 1–20 with max 4 per sector; if fewer
   than 10 fit, fill from the unused ranks 1–20 ignoring the sector limit; never pick worse than rank 20 (rest stays cash).
   Weights ∝ 1/63-day volatility; fills at the next open. If QQQ is at/below its 200-day average, all weights are halved.
2. **Mon/Wed** close: if a stock not held is in the top 3 and a holding is below rank 15, swap them (same dollars, any sector);
   then sell any holding worse than rank 30 (cash until Friday).
3. **Earnings rule (E5, from 2026-09-25):** a stock that is NOT held is not bought if decision date < next earnings date ≤ decision
   date + 5 calendar days (`Reports/earnings_date.csv`). Its slot goes to the next eligible stock in ranks 1–20, else cash; a mid-week
   top-3 candidate with earnings that close is skipped. Held stocks are never sold because of earnings. Reason in the logs:
   "earnings in N days (Wed Sep 30): not bought".

**Revert a rule** (edit `WINNER` in `backtest_engine.py`, then `python run_all.py`): earnings rule `earnings_block_days = None`;
rank-30 exit `midweek_exit_below = None`; mid-week swap `midweek_swap = None`; ranks 1–20 picks `max_pick_rank = None, cap_soft = False`;
universe `sector_mapping.EXPANDED_UNIVERSE = "high_beta_91"` (91) or `None` (78). Tracking rows keep the label they were written with.

**Backtest** (`backtest.ipynb`, 2022-04 → 2026-09, 0.1% per side): without the earnings rule +414.1%, Sharpe 1.37, max DD −32.2%,
never-seen (2022-04 → 2024-09) Sharpe 0.60. The earnings rule can only be backtested from late 2024 (earnings dates on disk) and was a
user decision, not a tested rule. The stock list is hand-picked with hindsight: expect weaker live results.

**Columns:** `Strategy_Weight` = decisions in force; `Provisional_Weight` = a full rebalance at the latest close; `Midweek_Check` = 1 on
Mon/Wed check sessions; `final_trade` BUY = held, SELL = not held and score < 0, HOLD = not held and score ≥ 0 (QQQ = BENCHMARK).
**Outputs:** `strategy_picks.csv`, `strategy_changes.csv` (adds / drops / earnings skips with reasons), `strategy_decisions.csv` (full
history), `strategy_midweek_check.csv` (this week in plain English), `strategy_holdings.csv`, `strategy_tracking.csv` (forward log vs QQQ/SPY).

### 0. Setup
Imports, paths and the live rule settings (`backtest_engine.WINNER`).

In [1]:
import importlib
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import backtest_engine as be
import sector_mapping

importlib.reload(sector_mapping)
importlib.reload(be)

REPORTS_DIR = str(sector_mapping.REPORTS_DIR)  # absolute path next to sector_mapping.py (notebook-safe)
SIGNAL_CSV = os.path.join(REPORTS_DIR, "signal_analysis.csv")
FACTOR_CSV = os.path.join(REPORTS_DIR, "factor_history.csv")
PICKS_CSV = os.path.join(REPORTS_DIR, "strategy_picks.csv")
CHANGES_CSV = os.path.join(REPORTS_DIR, "strategy_changes.csv")
HOLDINGS_CSV = os.path.join(REPORTS_DIR, "strategy_holdings.csv")
TRACKING_CSV = os.path.join(REPORTS_DIR, "strategy_tracking.csv")
DECISIONS_CSV = os.path.join(REPORTS_DIR, "strategy_decisions.csv")   # full weekly decision history (app ticker chart)
BENCH_CSV = os.path.join(REPORTS_DIR, "benchmark_prices.csv")          # SPY / QQQ / sector ETF closes (app RS lines)
TRACKING_START = "2026-09-18"   # first live rebalance decision of the ranking strategy
RULES_VERSION = (("v4-mw30" if be.WINNER.get("midweek_exit_below") else "v4-mw") if be.WINNER.get("midweek_swap") else "v3") \
    + ("-t20" if be.WINNER.get("max_pick_rank") else "") + (f"-e{be.WINNER['earnings_block_days']}" if be.WINNER.get("earnings_block_days") else "")
# v4-mw: weights also change at Mon/Wed swaps; v4-mw30: ... and Mon/Wed exits to cash (rank worse than 30); -t20: picks from ranks 1-20;
# -e5: no new buys with earnings in the next 5 days
S = be.WINNER

tradable = list(sector_mapping.tradable_symbols)                                # 96 stocks (C6-U96; 91 if EXPANDED_UNIVERSE = "high_beta_91", 78 if None)
display_only = [s for s in sector_mapping.stock_symbols if s not in tradable]  # QQQ: charted, never traded
print(f"{len(tradable)} tradable + display-only {display_only}; strategy: {S['name']}")

96 tradable + display-only ['QQQ']; strategy: C6: weekly top-10 ranking, max 4 per sector + soft QQQ regime (expanded universe, 96 stocks) [picks from ranks 1-20 only; sector cap relaxed to fill the 10 slots; top-3 swaps ignore the cap] + mid-week swap (Mon/Wed close: top 3 in, below rank 15 out) + mid-week exit (sell if worse than rank 30, cash until Friday) + no new buys with earnings in the next 5 days


### 1. Fetch bars (market data only) and compute point-in-time technicals

In [2]:
start = (datetime.now() - timedelta(days=1100)).date()   # ~2.3 years of output after the 200-bar warm-up
bars = be.fetch_daily_bars(tradable + be.BENCHMARKS + be.SECTOR_ETFS, start=start)
bars, dropped_partial = be.drop_partial_last_bar(bars)   # decisions are made on completed daily bars only
tech = be.build_technical(bars, symbols=tradable + display_only)
print(f"✅ Bars through {bars['Date'].max():%Y-%m-%d} (partial intraday bar dropped: {dropped_partial})")
print(f"   Missing: {sorted(set(tradable + display_only) - set(tech['Symbol']))}")

✅ Bars through 2026-09-24 (partial intraday bar dropped: False)
   Missing: []


### 2. Relative strength, strategy score, weekly top-10 portfolio + Mon/Wed mid-week swaps

The score uses price bars only (technicals + relative strength), so Mon/Wed runs need no news calls.

In [3]:
close_all = be.wide(bars, "Close")
dates = close_all.index
score_tech = be.wide(tech, "Technical_Score").reindex(index=dates, columns=tradable)
elig = be.bool_wide(tech, "eligible", dates, tradable)
rs_score, sector_rs = be.relative_strength(close_all, tradable)
strategy_score = S["w_tech"] * score_tech + (1 - S["w_tech"]) * rs_score
vol63 = close_all[tradable].pct_change().rolling(63).std()
regime = be.regime_series(close_all, S["regime_symbol"])
weekly = be.weekly_rebalance_days(dates, live=True)
rank_args = be.winner_rank_args(regime)
decisions, checks_log = [], []
# weekly rank_targets + (WINNER["midweek_swap"]) Mon/Wed swaps + (WINNER["midweek_exit_below"]) Mon/Wed exits to cash
# + (WINNER["earnings_block_days"]) no new buys with earnings in the next N days; midweek = check sessions
weights, midweek = be.winner_targets(strategy_score, elig, vol63, regime, weekly, tiebreak_w=rs_score,
                                     decision_log=decisions, check_log=checks_log)
# "if rebalanced at the latest close" preview; the earnings rule (WINNER["earnings_block_days"]) uses the real holdings
prov_log = []
earn_block = (be.earnings_days_ahead(dates, tradable, be.load_earnings(), S["earnings_block_days"])
              if S.get("earnings_block_days") else None)
provisional = be.rank_targets(strategy_score, elig, vol63, rebalance_days=weekly | (dates == dates[-1]),
                              decision_log=prov_log, tiebreak_w=rs_score, buy_block=earn_block,
                              held_w=weights.shift(1).fillna(0.0), **rank_args)
# Rank per date among eligible symbols only (1 = best). Ties (equal score at 6 decimals) -> higher RS_Score, then A-Z;
# the same order rank_targets walks down, so Strategy_Rank == the decision-log rank for every qualifying name.
strategy_rank = be.deterministic_rank(strategy_score, elig, tiebreak_w=rs_score)

def to_long(frame, name):
    return frame.stack(future_stack=True).rename(name).rename_axis(["Date", "Symbol"]).reset_index()

df = tech[tech["ma_200"] > 0].copy()
for frame, name in [(rs_score, "RS_Score"), (strategy_score.where(elig), "Strategy_Score"), (strategy_rank, "Strategy_Rank"),
                    (weights, "Strategy_Weight"), (sector_rs, "Sector Score")]:
    df = df.merge(to_long(frame, name), on=["Date", "Symbol"], how="left")
latest_date = df["Date"].max()
df = df.merge(to_long(provisional.loc[[latest_date]], "Provisional_Weight"), on=["Date", "Symbol"], how="left")
df["Regime_On"] = df["Date"].map(regime.astype(int)).fillna(0).astype(int)
df["Rebalance_Day"] = df["Date"].map(weekly.astype(int)).fillna(0).astype(int)
df["Midweek_Check"] = df["Date"].map(midweek.astype(int)).fillna(0).astype(int)
df["combined_signal"] = df["Strategy_Score"]          # the app ranks/plots combined_signal
last_rebalance = dates[weekly.to_numpy()][-1]
swap_days = sorted({c["Date"] for c in checks_log if c["Action"] in ("SWAP", "SELL")})   # mid-week days that changed holdings
last_decision = max([last_rebalance] + [d for d in swap_days if d > last_rebalance])   # latest decision in force
print(f"✅ Latest {latest_date:%Y-%m-%d} | last weekly rebalance {last_rebalance:%Y-%m-%d} | "
      f"regime ({S['regime_symbol']} > 200d MA): {'ON' if regime.iloc[-1] else 'OFF (weights x ' + str(S['regime_scale']) + ')'}")
print(f"   Mid-week swap: {S.get('midweek_swap')}, exit below rank: {S.get('midweek_exit_below')} | check days in window {int(midweek.sum())}, "
      f"swaps {len([c for c in checks_log if c['Action'] == 'SWAP'])}, exits {len([c for c in checks_log if c['Action'] == 'SELL'])}"
      f" | last decision {last_decision:%Y-%m-%d}")
print("   Current holdings:", weights.loc[latest_date][lambda w: w > 0].sort_values(ascending=False).round(3).to_dict())

✅ Latest 2026-09-24 | last weekly rebalance 2026-09-18 | regime (QQQ > 200d MA): ON
   Mid-week swap: {'enter_top': 3, 'exit_below': 15, 'days': ['Mon', 'Wed']}, exit below rank: 30 | check days in window 314, swaps 81, exits 43 | last decision 2026-09-21
   Current holdings: {'MRK': 0.16, 'FTNT': 0.128, 'GTLB': 0.104, 'ANET': 0.101, 'TWLO': 0.09, 'RBRK': 0.079, 'HOOD': 0.078, 'TEM': 0.062, 'TEAM': 0.06}


### 3. Fundamentals & sentiment (display only) + daily factor snapshot

No history exists for these factors, so they are not part of the tested strategy. They are shown for the latest date only and
appended every run to `Reports/factor_history.csv` (one row per symbol per bar date) so they can be backtested later.
`Sector Score` is now the sector ETF's 63-day return minus SPY's (%, point-in-time) instead of the 7-day ETF return × 3.
`Legacy_Combined` = the old 60/25/10/5 blend (latest date only, for comparison).

In [4]:
sentiment = pd.read_csv(os.path.join(REPORTS_DIR, "weighted_sentiment.csv"), usecols=["Symbol", "SentimentScore"])
fundamentals = pd.read_csv(os.path.join(REPORTS_DIR, "balance_sheet_weights.csv"), usecols=["Symbol", "Fundamental_Weight"])
etf_close = close_all[list(sector_mapping.sector_etfs)]
etf_close = etf_close[etf_close.index >= latest_date - pd.Timedelta(days=7)]
sector_7d = ((etf_close.iloc[-1] / etf_close.iloc[0] - 1) * 100).rename(index=sector_mapping.sector_etfs)
legacy_sector = pd.Series({s: sector_7d.get(sector_mapping.symbol_sector.get(s), 0.0) * 3 for s in df["Symbol"].unique()})

is_latest = df["Date"].eq(latest_date)
df = df.merge(sentiment, on="Symbol", how="left").merge(fundamentals, on="Symbol", how="left")
# No relevant news = neutral sentiment (0), not the average of other stocks; missing fundamentals stay empty ("—" in the app)
df.loc[is_latest, "SentimentScore"] = df.loc[is_latest, "SentimentScore"].fillna(0.0)
df.loc[~is_latest, ["SentimentScore", "Fundamental_Weight"]] = np.nan
df["Legacy_Combined"] = np.where(is_latest, (
    0.60 * df["Technical_Score"] + 0.25 * (df["Fundamental_Weight"].fillna(0).clip(-10, 10) * 10.0)
    + 0.10 * (df["Symbol"].map(legacy_sector).clip(-30, 30) / 30.0 * 100.0) + 0.05 * (df["SentimentScore"].clip(-10, 10) * 10.0)
), np.nan)

snap = df.loc[is_latest, ["Date", "Symbol", "SentimentScore", "Fundamental_Weight", "Sector Score", "RS_Score",
                          "Technical_Score", "Strategy_Score"]].rename(columns={"Date": "bar_date", "Sector Score": "Sector_RS_63"})
snap.insert(0, "as_of", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
snap["Sector_Score_7d_legacy"] = snap["Symbol"].map(legacy_sector)
history = pd.read_csv(FACTOR_CSV, parse_dates=["bar_date"]) if os.path.exists(FACTOR_CSV) else pd.DataFrame()
history = pd.concat([history, snap], ignore_index=True).drop_duplicates(["bar_date", "Symbol"], keep="last")
history.round(4).to_csv(FACTOR_CSV, index=False)
print(f"✅ factor_history.csv: {len(history)} rows, {history['bar_date'].nunique()} bar dates")

✅ factor_history.csv: 97 rows, 1 bar dates


### 4. BUY / SELL / HOLD + streaks

- **final_trade**: BUY = held in the current top-10 portfolio (after any mid-week swap); SELL = not held and Strategy_Score < 0; HOLD = otherwise.
- Rows saved under rules v2 keep their final_trade (except the last saved date). Older rows (old rules) are recomputed.
- **Streaks**: consecutive days with the same final_trade.

In [5]:
trade_df = df.sort_values(["Symbol", "Date"]).reset_index(drop=True)
rule = pd.Series("HOLD", index=trade_df.index)
rule = rule.mask(trade_df["Strategy_Score"] < S["min_score"], "SELL").mask(trade_df["Strategy_Weight"] > 0, "BUY")
rule = rule.mask(trade_df["Symbol"].isin(display_only), "BENCHMARK")

prev = pd.read_csv(SIGNAL_CSV) if os.path.exists(SIGNAL_CSV) else pd.DataFrame()
frozen = 0
if "rules_version" in prev.columns:
    prev["Date"] = pd.to_datetime(prev["Date"]).dt.normalize()
    prev = prev[(prev["rules_version"] == RULES_VERSION) & (prev["Date"] < prev["Date"].max())]
    prev = prev.drop_duplicates(["Symbol", "Date"], keep="last")[["Symbol", "Date", "final_trade"]]
    trade_df = trade_df.merge(prev.rename(columns={"final_trade": "prev_trade"}), on=["Symbol", "Date"], how="left")
    frozen = int(trade_df["prev_trade"].notna().sum())
    trade_df["final_trade"] = trade_df.pop("prev_trade").fillna(rule)
else:
    trade_df["final_trade"] = rule

new_run = trade_df["final_trade"].ne(trade_df.groupby("Symbol")["final_trade"].shift())
run_length = trade_df.groupby(new_run.cumsum()).cumcount() + 1
for trade, col in [("BUY", "Buy Streak"), ("SELL", "Sell Streak"), ("HOLD", "Hold Streak")]:
    trade_df[col] = run_length.where(trade_df["final_trade"].eq(trade), 0)
print(f"✅ Streaks for {trade_df['Symbol'].nunique()} symbols (frozen v2 rows: {frozen})")
print(f"   latest final_trade counts: {trade_df.loc[trade_df['Date'] == latest_date, 'final_trade'].value_counts().to_dict()}")

✅ Streaks for 97 symbols (frozen v2 rows: 0)
   latest final_trade counts: {'SELL': 45, 'HOLD': 42, 'BUY': 9, 'BENCHMARK': 1}


### 5. Earnings flags, save signals + current picks

Symbols with earnings in the next 2 days get final_trade `EARNING` on the latest date (label only: the tested strategy holds through
earnings, so `Strategy_Weight` is unchanged). `Reports/strategy_picks.csv` lists the current portfolio and provisional picks.

In [6]:
OUTPUT_COLUMNS = [
    "Date", "Symbol", "Close", "ma_10", "ma_30", "ma_50", "ma_100", "ma_200",
    "RSI", "macd", "MACD Signal", "Technical_Score", "SentimentScore", "Fundamental_Weight",
    "Sector Score", "combined_signal", "final_trade", "Buy Streak", "Sell Streak", "Hold Streak",
    "is_earnings_date",
    # v2 additions
    "RS_Score", "Strategy_Score", "Strategy_Rank", "Strategy_Weight", "Provisional_Weight", "Regime_On", "Rebalance_Day",
    "Legacy_Combined", "rules_version",
    # v4-mw
    "Midweek_Check",
]

earnings = be.load_earnings()
today = pd.Timestamp.now(tz=be.EASTERN).tz_localize(None).normalize()          # exchange date, not the Mac's local date
window_end = be.next_sessions(today - pd.Timedelta(days=1), 3)[-1]             # today (or next session) + 2 more sessions
near_earnings = earnings.loc[earnings["Earnings Date"].between(today, window_end), "Symbol"]

out = trade_df.rename(columns={"rsi": "RSI", "macd_signal": "MACD Signal"})
out["rules_version"] = RULES_VERSION
out["is_earnings_date"] = pd.MultiIndex.from_frame(out[["Symbol", "Date"]]).isin(
    pd.MultiIndex.from_frame(earnings[["Symbol", "Earnings Date"]])
).astype(int)
out.loc[(out["Date"] == latest_date) & out["Symbol"].isin(near_earnings),
        ["final_trade", "Buy Streak", "Sell Streak", "Hold Streak"]] = ["EARNING", 0, 0, 0]

out = out[OUTPUT_COLUMNS].round(4).sort_values(["Symbol", "Date"], ascending=[True, False])
out.to_csv(SIGNAL_CSV, index=False)

latest = out[out["Date"] == latest_date].set_index("Symbol")
next_ed = earnings[earnings["Earnings Date"] >= today].sort_values("Earnings Date").drop_duplicates("Symbol").set_index("Symbol")["Earnings Date"]
picks = latest[(latest["Strategy_Weight"] > 0) | (latest["Provisional_Weight"] > 0)].copy()
picks = picks.assign(Sector=picks.index.map(sector_mapping.symbol_sector), Next_Earnings=picks.index.map(next_ed),
                     Held=(picks["Strategy_Weight"] > 0).astype(int), As_Of=latest_date, Last_Rebalance=last_rebalance,
                     Last_Decision=last_decision,
                     Regime_On=int(regime.iloc[-1]), Strategy=S["name"])
picks = picks.reset_index()[["As_Of", "Last_Rebalance", "Last_Decision", "Strategy", "Regime_On", "Symbol", "Sector", "Held", "Strategy_Weight",
                             "Provisional_Weight", "Strategy_Score", "Strategy_Rank", "Technical_Score", "RS_Score", "Close",
                             "Next_Earnings"]].sort_values(["Held", "Strategy_Weight", "Strategy_Score"], ascending=False)
picks.round(4).to_csv(PICKS_CSV, index=False)

# --- What changed at the latest decision (weekly rebalance or mid-week swap) and what a full rebalance now would change ---
dec = pd.DataFrame(decisions)
if len(dec):
    dec["Check"] = np.where(dec["Date"].isin(dates[midweek.to_numpy(bool)]), "mid-week", "weekly")
changes = dec[dec["Date"] == last_decision].assign(View="last rebalance") if len(dec) else pd.DataFrame()   # view name kept for the app
prov = pd.DataFrame(prov_log)
if len(prov) and latest_date != last_rebalance:
    pv = prov[prov["Date"] == latest_date].copy()
    held_now = weights.loc[latest_date]
    for sym in held_now[held_now > 0].index.difference(pv["Symbol"]):   # held (e.g. swapped in) but not reached by the walk-down
        pv = pd.concat([pv, pd.DataFrame([{"Date": latest_date, "Symbol": sym, "Status": "drop", "Rank": strategy_rank.at[latest_date, sym],
                                           "Reason": f"rank {strategy_rank.at[latest_date, sym]:.0f} outside top {S['n']}",
                                           "Score": strategy_score.at[latest_date, sym], "Sector": sector_mapping.symbol_sector.get(sym),
                                           "New_Weight": 0.0}])], ignore_index=True)
    pv["Old_Weight"] = pv["Symbol"].map(held_now).fillna(0.0)          # compare with what is held now (after any swap)
    pv["Status"] = np.select([(pv["New_Weight"] > 0) & (pv["Old_Weight"] > 0), pv["New_Weight"] > 0, pv["Old_Weight"] > 0],
                             ["hold", "add", "drop"], np.where(pv["Status"].isin(["add", "drop", "hold"]), "not selected", pv["Status"]))
    changes = pd.concat([changes, pv.assign(View="if rebalanced at latest close")])
if len(changes):
    changes = changes[changes["Status"].isin(["add", "drop", "hold"]) | changes["Reason"].str.startswith(("skipped", "earnings in"))]
    changes.round(4).to_csv(CHANGES_CSV, index=False)
    print("Changes at last rebalance:", changes[changes["View"] == "last rebalance"].groupby("Status")["Symbol"].apply(list).to_dict())

# --- Full decision history: every add / hold / drop and sector-cap skip at each weekly rebalance + mid-week swap (point-in-time) ---
if len(dec):
    dec_hist = dec[dec["Status"].isin(["add", "drop", "hold"]) | dec["Reason"].str.startswith(("skipped", "earnings in"))].copy()
    dec_hist["Regime_On"] = dec_hist["Date"].map(regime.astype(int))
    dec_hist.round(4).to_csv(DECISIONS_CSV, index=False)
    print(f"strategy_decisions.csv: {len(dec_hist)} rows, {dec_hist['Date'].nunique()} rebalances")

# --- Mid-week swap check: this week's decisions in plain English (Friday rebalance + Mon/Wed checks) ---
MIDWEEK_CSV = os.path.join(REPORTS_DIR, "strategy_midweek_check.csv")
MW = S.get("midweek_swap")

def _day(d):
    return f"{pd.Timestamp(d):%a %b} {pd.Timestamp(d).day}"

def earnings_skip_text(rows):
    """' Not bought (earnings within 5 days): MU rank 4 (earnings Wed Sep 30).' for a decision's log rows, or ''."""
    e = rows[rows["Reason"].astype(str).str.startswith("earnings in")] if len(rows) else rows
    if not len(e):
        return ""
    items = [f"{r.Symbol} rank {r.Rank:.0f} ({r.Reason.split(': not bought')[0]})"
             for r in e.sort_values("Rank").itertuples()]
    return f" Not bought (earnings within {S['earnings_block_days']} days): " + "; ".join(items) + "."

def _fill_after(d):
    later = dates[dates > d]
    return later[0] if len(later) else be.next_sessions(d, 1)[0]

nxt_d, nxt_kind, nxt_fill = be.next_decision(latest_date)
next_msg = f"Next decision: {nxt_kind} at the {_day(nxt_d)} close, trades at the {_day(nxt_fill)} open."
rb = dec[dec["Date"] == last_rebalance] if len(dec) else pd.DataFrame(columns=["Status"])
f0 = _fill_after(last_rebalance)
mw_rows = [{"Event": "full rebalance", "Event_Date": last_rebalance, "Applies_To_Open": f0, "Action": "REBALANCE",
            "Sell": ", ".join(sorted(rb.loc[rb["Status"] == "drop", "Symbol"])), "Sell_Rank": np.nan,
            "Buy": ", ".join(sorted(rb.loc[rb["Status"] == "add", "Symbol"])), "Buy_Rank": np.nan, "Weight_%": np.nan,
            "Message": (f"Full rebalance at the {_day(last_rebalance)} close: trade to the new top {S['n']} at the {_day(f0)} open "
                        f"({(rb['Status'] == 'add').sum()} buy, {(rb['Status'] == 'drop').sum()} sell, {(rb['Status'] == 'hold').sum()} keep)."
                        + earnings_skip_text(rb))}]
for c_ in [c for c in checks_log if c["Date"] > last_rebalance]:
    f1 = _fill_after(c_["Date"])
    if c_["Action"] == "SWAP":
        srank = f"rank {c_['Sell_Rank']:.0f}" if c_["Sell_Rank"] == c_["Sell_Rank"] else "no longer qualifies: score <= 0 or no data"
        msg = (f"Mid-week check at the {_day(c_['Date'])} close: SELL {c_['Sell']} ({srank}) and BUY {c_['Buy']} "
               f"(rank {c_['Buy_Rank']:.0f}) at the {_day(f1)} open, same dollar amount.")
    elif c_["Action"] == "SELL":                 # mid-week exit (WINNER["midweek_exit_below"])
        srank = f"rank {c_['Sell_Rank']:.0f}" if c_["Sell_Rank"] == c_["Sell_Rank"] else "no longer qualifies: score <= 0 or no data"
        msg = (f"Mid-week check at the {_day(c_['Date'])} close: SELL {c_['Sell']} ({srank}, worse than {S['midweek_exit_below']}) "
               f"at the {_day(f1)} open, hold the cash until the Friday rebalance.")
    else:
        msg = f"Mid-week check at the {_day(c_['Date'])} close: No swap ({c_['Note']}). Nothing to trade at the {_day(f1)} open."
    mw_rows.append({"Event": "mid-week check", "Event_Date": c_["Date"], "Applies_To_Open": f1, "Action": c_["Action"],
                    "Sell": c_["Sell"], "Sell_Rank": c_["Sell_Rank"], "Buy": c_["Buy"], "Buy_Rank": c_["Buy_Rank"],
                    "Weight_%": c_["Weight"] * 100 if c_["Weight"] == c_["Weight"] else np.nan, "Message": msg})
mwc = pd.DataFrame(mw_rows)
mwc.insert(0, "As_Of", latest_date)
mwc["Is_Latest"] = (mwc["Event_Date"] == mwc["Event_Date"].max()).astype(int)
mwc["Status"] = np.select([mwc["Action"] == "NO SWAP", mwc["Applies_To_Open"] > latest_date],
                          ["no trade", "to do at the next open"], "done (that open has passed)")
mwc["Next_Decision"], mwc["Next_Decision_Type"], mwc["Next_Fill"], mwc["Next_Message"] = nxt_d, nxt_kind, nxt_fill, next_msg
mwc["Midweek_Rule"] = ((f"Mon/Wed close: top {MW['enter_top']} in" + (" (any sector)" if S.get("cap_soft") else "")
                        + f", below rank {MW['exit_below']} out"
                        + (f"; then sell any holding worse than rank {S['midweek_exit_below']} (cash until Friday)"
                           if S.get("midweek_exit_below") else "")) if MW else "off (weekly only)")
mwc["Rules"] = S.get("tag", "C6")
mwc.round(4).to_csv(MIDWEEK_CSV, index=False)
print("Mid-week swap check:")
for m_ in mwc["Message"]:
    print("  " + m_)
print("  " + next_msg)

# --- Benchmark and sector-ETF closes (for relative-strength lines in the app) ---
bench_cols = [c for c in be.BENCHMARKS + be.SECTOR_ETFS if c in close_all.columns]
(close_all.loc[close_all.index >= df["Date"].min(), bench_cols].round(4).rename_axis("Date").reset_index()
 .to_csv(BENCH_CSV, index=False))

# --- Per-holding risk ---
open_all = be.wide(bars, "Open")
atr_w = be.wide(tech, "atr").reindex(index=dates, columns=tradable)
holdings = be.holding_details(weights, open_all, close_all, atr_w, vol63, k=S["atr_stop_k"])
holdings.round(4).to_csv(HOLDINGS_CSV, index=False)

# --- Forward-tracking log vs QQQ / SPY since the first live rebalance (past rows are never rewritten) ---
track = be.forward_tracking(weights, open_all, close_all, TRACKING_START, rebalance=weekly)
if len(track):
    track["Rules"] = S.get("tag", "C6")
    if os.path.exists(TRACKING_CSV):
        old = pd.read_csv(TRACKING_CSV, parse_dates=["Date"])
        if "Rules" not in old.columns:
            old["Rules"] = "C6"                      # rows written before the Rules column existed were produced by C6 (max 4 per sector)
        new = track[track["Date"] > old["Date"].max()].copy()
        base = track[track["Date"] == old["Date"].max()]
        if len(new) and len(base):                   # chain-link: a rules change continues the curve instead of jumping
            for c in ("Strategy", "QQQ", "SPY"):
                new[c] = new[c] / base[c].iloc[0] * old[c].iloc[-1]
        track = pd.concat([old, new], ignore_index=True)
    track.round(6).to_csv(TRACKING_CSV, index=False)
    print(f"Tracking since {TRACKING_START}: strategy {(track['Strategy'].iloc[-1] - 1) * 100:+.2f}%, "
          f"QQQ {(track['QQQ'].iloc[-1] - 1) * 100:+.2f}%, SPY {(track['SPY'].iloc[-1] - 1) * 100:+.2f}% ({len(track)} sessions)")

print(f"Earnings in next 2 sessions: {sorted(near_earnings.unique())}")
print(f"✅ Saved {len(out)} rows × {len(out.columns)} columns, {out['Symbol'].nunique()} symbols, latest {latest_date:%Y-%m-%d}")
picks

Changes at last rebalance: {'drop': ['APA'], 'hold': ['ANET', 'FTNT', 'GTLB', 'HOOD', 'MRK', 'TEAM', 'TWLO', 'TEM', 'RBRK']}
strategy_decisions.csv: 2979 rows, 219 rebalances
Mid-week swap check:
  Full rebalance at the Fri Sep 18 close: trade to the new top 10 at the Mon Sep 21 open (5 buy, 4 sell, 5 keep).
  Mid-week check at the Mon Sep 21 close: SELL APA (rank 34, worse than 30) at the Tue Sep 22 open, hold the cash until the Friday rebalance.
  Mid-week check at the Wed Sep 23 close: No swap (all top-3 stocks are already held; no holding is worse than rank 30). Nothing to trade at the Thu Sep 24 open.
  Next decision: full rebalance at the Fri Sep 25 close, trades at the Mon Sep 28 open.
Tracking since 2026-09-18: strategy +2.13%, QQQ +1.71%, SPY +0.02% (4 sessions)
Earnings in next 2 sessions: []
✅ Saved 51706 rows × 31 columns, 97 symbols, latest 2026-09-24


,As_Of,Last_Rebalance,Last_Decision,Strategy,Regime_On,Symbol,Sector,Held,Strategy_Weight,Provisional_Weight,Strategy_Score,Strategy_Rank,Technical_Score,RS_Score,Close,Next_Earnings
7,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,MRK,Health Care,1,0.16,0.00,35.29,30.00,60.29,10.28,147.98,2026-10-29
3,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,FTNT,Technology,1,0.13,0.00,57.49,9.00,64.71,50.28,178.67,2026-11-03
4,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,GTLB,Technology,1,0.10,0.00,61.95,7.00,60.29,63.61,48.62,2026-12-07
1,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,ANET,Technology,1,0.10,0.00,46.96,16.00,63.24,30.69,205.70,2026-11-02
12,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,TWLO,Technology,1,0.09,0.09,66.98,2.00,66.18,67.78,299.66,2026-10-28
9,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,RBRK,Technology,1,0.08,0.09,67.61,1.00,69.12,66.11,113.80,NaT
5,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,HOOD,Financials,1,0.08,0.09,44.84,19.00,69.12,20.56,120.82,2026-11-03
11,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,TEM,Health Care,1,0.06,0.07,58.53,8.00,64.71,52.36,82.24,2026-11-02
10,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,TEAM,Technology,1,0.06,0.07,65.20,3.00,64.71,65.69,192.61,2026-10-26
0,2026-09-24,2026-09-18,2026-09-21,"C6: weekly top-10 ranking, max 4 per sector + ...",1,AMD,Technology,0,0.00,0.09,64.60,4.00,70.59,58.61,629.26,2026-11-02
